In [1]:
import pandas as pd
import random
import time
import warnings
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

In [2]:
def load_random_hmda_sample(target_rows=1_000_000, chunk_size=10_000):
    years = ['2020', '2021', '2022', '2023']
    states = ['AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA',
              'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD',
              'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ',
              'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC',
              'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY']

    df_accum = pd.DataFrame()
    
    # Initialize progress bar
    pbar = tqdm(total=target_rows, unit=' rows', desc='Downloading', 
                bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} rows [{elapsed}<{remaining}]',
                miniters=1, mininterval=0.1)
    
    while len(df_accum) < target_rows:
        year = random.choice(years)
        state = random.choice(states)
        url = f"https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?years={year}&states={state}"
        
        try:
            df = pd.read_csv(url, low_memory=False)
        
            if len(df) > chunk_size:
                df = df.sample(n=chunk_size, random_state=random.randint(1, chunk_size))
            df_accum = pd.concat([df_accum, df], ignore_index=True)
            
            # Update progress bar 
            pbar.update(len(df))
            
            time.sleep(1) 
        except Exception as e:
            pbar.write(f"Error loading {state}-{year}: {e}")
            continue
    
    # Close progress bar
    pbar.close()
    
    # Save downloaded data to CSV 
    print(f"\nSaving {len(df_accum):,} rows to CSV...")
    df_accum.to_csv("hmda_random_sample_2020_2023.csv", index=False)
    print("Download complete!")

In [3]:
load_random_hmda_sample()

Downloading:   0%|          | 0/1000000 rows [00:00<?]


Saving 1,000,000 rows to CSV...
Download complete!
